# Lesson 1 — Introduction to Optimization

**Track:** Basic &nbsp;&nbsp;|&nbsp;&nbsp; **Estimated time:** 90 minutes

## Learning objectives

After this lesson you will be able to:

1. State the standard form of a mathematical optimization problem.
2. Distinguish LP, IP/MILP, NLP, and MINLP.
3. Recognize when an "optimization problem" in the wild is in fact an LP, an
   MILP, or a (possibly nonconvex) MINLP.
4. Solve a small LP and a small MILP with `discopt` and read the output.
5. Cite the canonical references for the field {cite:p}`Bertsimas1997,Wolsey1998,Boyd2004`.


## 1. Standard form

A mathematical optimization problem (or **mathematical program**) is

$$
\begin{aligned}
\min_{x \in \mathbb{R}^n}\quad & f(x) \\
\text{s.t.}\quad & g_i(x) \le 0, \quad i = 1, \dots, m \\
& h_j(x) = 0, \quad j = 1, \dots, p \\
& x \in X
\end{aligned}
$$

- $x$ are **decision variables**.
- $f$ is the **objective**.
- $g_i, h_j$ are **constraints**.
- $X$ encodes set membership constraints (e.g., $x \in \mathbb{Z}^n$ for
  integer variables, $x_i \in \{0,1\}$ for binaries).

The problem class is determined by the structure of $f$, $g_i$, $h_j$, and $X$:

| Class | $f, g, h$ | $X$ |
|-------|-----------|-----|
| LP    | linear (affine) | $\mathbb{R}^n_{\ge 0}$ usually |
| IP/MILP | linear | $\mathbb{Z}^n$ or mixed |
| NLP   | nonlinear (smooth) | continuous |
| MINLP | nonlinear | mixed integer |

`discopt` is built for **MINLP**, the most general of these — every other
class is a special case it can handle by recognizing structure
{cite:p}`Belotti2013,Tawarmalani2005`.


## 2. Why this matters

Optimization is the lingua franca of decision making under constraints. In
operations, you optimize schedules, routing, allocation
{cite:p}`Williams2013`. In engineering design, you optimize structures and
processes {cite:p}`Biegler2010`. In machine learning, training itself is
optimization. Markowitz's portfolio model {cite:p}`Markowitz1952` is an LP/QP;
Stigler's diet problem {cite:p}`Stigler1945` is the canonical LP.

Three observations animate the entire course:

- **Convexity is the watershed.** Convex problems have global optima
  reachable in polynomial time {cite:p}`Boyd2004`. Nonconvex problems may
  have many local optima, and finding the global one is hard.
- **Integer variables make a problem combinatorial.** MILP is NP-hard
  {cite:p}`Wolsey1998`, even when the LP relaxation is trivial.
- **Structure is exploitable.** A solver that recognizes a problem is in
  fact an LP, or convex, or has totally-unimodular constraint matrix, can be
  *vastly* faster than a generic MINLP solver.


## 3. A first LP — the diet problem

We pick quantities $x_1, x_2, x_3$ of three foods (say bread, milk, beans) to
minimize cost while meeting nutritional requirements. From
{cite:t}`Stigler1945`:

$$
\begin{aligned}
\min_{x \ge 0}\quad & c^\top x \\
\text{s.t.}\quad & A x \ge b
\end{aligned}
$$

where $c \in \mathbb{R}^3$ are unit costs, $A_{ij}$ is nutrient $i$ per unit
of food $j$, and $b_i$ is the daily requirement of nutrient $i$. Let's solve
it with `discopt`.


In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np
import discopt as do

# Costs ($/unit) for [bread, milk, beans]
c = np.array([0.50, 0.80, 0.30])

# Nutrients per unit: rows are [calories, protein, calcium]
A = np.array([
    [250, 120, 350],   # calories
    [  8,   8,  20],   # protein (g)
    [ 30, 300,  60],   # calcium (mg)
])
b = np.array([2000, 50, 800])  # daily requirements

m = do.Model("diet")
x = m.continuous("x", shape=(3,), lb=0)
m.subject_to(A @ x >= b)
m.minimize(c @ x)

result = m.solve()
print(result)
print("Foods purchased:", result.x)
print("Daily cost: $", round(result.objective, 2))


## 4. A first MILP — the knapsack

Now suppose we choose **whether** to take each item (yes/no) — a binary
decision. The 0/1 knapsack:

$$
\max_{x \in \{0,1\}^n} \; v^\top x \quad \text{s.t.} \quad w^\top x \le W
$$

This is the canonical NP-hard MILP. With $n = 5$ items it's trivial; with
$n = 100$, the LP relaxation gives only an upper bound and we need
branch-and-bound (Lesson 6) to close the gap.


In [ ]:
rng = np.random.default_rng(0)
n = 10
v = rng.integers(1, 20, size=n)   # values
w = rng.integers(1, 15, size=n)   # weights
W = int(0.4 * w.sum())            # capacity = 40 % of total

m = do.Model("knapsack")
x = m.binary("x", shape=(n,))
m.subject_to(w @ x <= W)
m.maximize(v @ x)
res = m.solve()

x_val = res.value(x)                          # ndarray (r.x is a dict by name)
print("Selected items:", np.where(x_val > 0.5)[0].tolist())
print("Total value:", res.objective, "Capacity used:", int(w @ x_val), "/", W)


## 5. A nonconvex NLP — and why it's harder

Consider minimizing the Six-Hump Camel function on a box:

$$
f(x_1, x_2) = (4 - 2.1 x_1^2 + x_1^4/3)\,x_1^2 + x_1 x_2 + (-4 + 4 x_2^2) x_2^2
$$

It has six local minima, two of them global at
$(\pm 0.0898, \mp 0.7126)$ with $f^\star \approx -1.0316$. A local NLP
solver started from a random point can land on any of the six. Spatial
branch-and-bound (Lesson 21) provably finds the best one
{cite:p}`Tawarmalani2005,Floudas1999`.

`discopt`'s solver auto-detects nonconvexity (`r.convex_fast_path` will be
`False` here) and applies its global routine; you don't choose `local` vs
`global` explicitly.


In [ ]:
from discopt.modeling import Model, sin, cos  # noqa: F401

m = Model("camel")
x1 = m.continuous("x1", lb=-3, ub=3)
x2 = m.continuous("x2", lb=-2, ub=2)
f = (4 - 2.1*x1**2 + x1**4/3)*x1**2 + x1*x2 + (-4 + 4*x2**2)*x2**2
m.minimize(f)

r = m.solve()
print(f"f* = {r.objective:.4f}   x = ({r.value(x1):.4f}, {r.value(x2):.4f})")
print(f"convex fast path: {r.convex_fast_path}   nodes: {r.node_count}")
# This single solve recovers ONE of the six local minima — Exercise 4 below
# asks you to find all six by sweeping initial points.


## 6. The taxonomy in one diagram

```
        +-------------------- Optimization --------------------+
        |                                                      |
       LP                                                  Nonlinear
        |                                                      |
        +-- IP / MILP                              +-- Convex NLP
        |                                          |
        +-- Network flow (TUM)                     +-- Nonconvex NLP
        |                                          |
        +-- Conic (SOCP, SDP)                      +-- MINLP (convex)
                                                   |
                                                   +-- MINLP (nonconvex)
```

We will visit every node of this tree.

## 7. What's next

- Lesson 2 dives into LP geometry and the simplex method.
- Lesson 3 walks through the `discopt` modeling API in detail.
- Exercise notebook for this lesson is in `exercises.ipynb`.

## References

The canonical textbook references for this lesson are
{cite:p}`Bertsimas1997` (LP), {cite:p}`Wolsey1998` (IP/MILP),
{cite:p}`Boyd2004` (convex), {cite:p}`Nocedal2006` (NLP), and
{cite:p}`Williams2013` (modeling). For the history,
{cite:t}`Dantzig1963,Kantorovich1960,Dorfman1958` are foundational.
